### feature = 문제 / 입력 데이터 / 모델이 보는 정보
### target  = 정답 / 예측해야 하는 값

## 하이퍼파라미터 기본 개념
하이퍼파라미터: 모델이 데이터에서 스스로 학습하는 값이 아니라, 사람이 학습 전에 미리 정하는 설정값임.

- alpha: 규제를 얼마나 강하게 적용할지 정하는 값임.
- l1_ratio: ElasticNet에서 L1 규제와 L2 규제를 어떤 비율로 섞을지 정하는 값임.

# Regularized Linear Models 규제 선형 회귀
다항식이 복잡해지져서 회귀계수가 매우 크게 설정이되면서 과대적합이 되고
평가데이터세트에 대해서 형편없는 예측 성능을 보이게 된다.

비용함수의 최소값을 구하는 것이 모델이 추구하는 목적이므로,
비용함수의 최소값을 구하는데 **𝝰 제약**을 걸어 과적합을 방지하는 것을 규제라고 한다.
- Lasso L1방식의 규제 적용
- Ridge L2방식의 규제 적용
- ElasticNet L1, L2규제를 결합한 모델. 특성이 많은 데이터셋에 적용. L1규제로 특성개수를 줄이고, L2규제로 계수값의 크기도 조정할 수 있다.


**비용함수 목표**

$
비용함수 목표 = \min \left(\text{RSS}(w) + \alpha \times W\right)
$

이 수식은 규제를 적용한 비용함수를 의미하며, 다음과 같은 요소들로 이루어져 있다:
1. **RSS(w)**: 이 부분은 Residual Sum of Squares의 약자로, 잔차 제곱합을 의미한다. 선형 회귀 모델에서 주로 사용하는 손실 함수로, 각 데이터 포인트에서의 예측 값과 실제 값 간의 차이를 제곱한 것들의 합이다. 즉, 모델의 예측 오차를 측정하는 부분이다. 수식으로는 다음과 같이 표현된다:
    $
    \text{RSS}(w) = \sum_{i=1}^n \left(y_i - \hat{y}_i\right)^2
    $
   여기서 $y_i$는 실제 값, $\hat{y}_i$는 모델의 예측 값이다.
2. **$\alpha$**: 규제 강도를 나타내는 하이퍼파라미터이다. 이 값이 클수록 규제의 효과가 커지고, 작을수록 규제의 효과가 줄어든다. 모델이 과적합되기 쉬운 경우, $\alpha$를 크게 설정하여 가중치를 제어할 수 있다.
3. **$W$**: 가중치들의 규제 항을 의미한다. 이는 가중치의 크기에 페널티를 부여하는 부분으로, 모델의 복잡도를 조절하는 역할을 한다.
   - L1 규제: $W = \sum_{j=1}^p |w_j|$
   - L2 규제: $W = \sum_{j=1}^p w_j^2$
수식을 다시 설명하면, 비용함수의 목표는 잔차 제곱합(RSS)과 규제 항($\alpha \times W$)을 더한 값을 최소화하는 것이다. 즉, 이 비용함수의 최적화 목표는 두 가지를 달성하고자 한다:
1. 모델의 예측 오차(RSS)를 줄이는 것.
2. 모델의 복잡도를 줄여서 가중치의 크기를 제어하는 것($\alpha \times W$).

**적합합 규제를 선택하려면 :**

1. **Lasso (L1 규제)**
  - 불필요한 피처를 자동으로 제거할 때 유용하다.
  - 많은 피처 중 일부만 중요할 때 사용하면 스파스한 모델을 만듦.
2. **Ridge (L2 규제)**
  - 모든 피처가 유의미하고 예측에 기여한다고 생각될 때 적합하다.
  - 피처가 많고 과적합을 방지하고 싶을 때 사용한다.
3. **Elastic Net (L1 + L2 규제)**
  - 피처를 일부 제거하면서도 나머지의 가중치도 줄이고 싶을 때 유용하다.
  - 상관관계가 높은 피처가 있을 때 선택을 안정적으로 한다.


## L2
- L2방식의 규제를 구현한 Ridge 클래스를 사용할 수 있다.
- 모든 피쳐의 회귀계수를 규제해 과적합을 방지한다.


## 실습 환경 준비

- NumPy: 배열 계산과 수치 연산을 다루기 위한 기본 라이브러리임.
- Pandas: 표 형태 데이터를 DataFrame으로 다루기 위한 라이브러리임.
- Matplotlib: 그래프를 그려 데이터 분포와 모델 결과를 시각화하는 라이브러리임.
- Seaborn: 통계 그래프를 더 쉽게 그리기 위한 시각화 라이브러리임.


In [1]:
# 초기 세팅용 import 구문입니다. 먼저 실행한 뒤 실습 코드를 작성합니다.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.dates import drange
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.metrics import mean_squared_error, mean_absolute_error, root_mean_squared_error
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet


## California Housing 데이터 불러오기

In [6]:
# 스켈레튼에 켈리포니아 하우징 데이터 불러오기
from sklearn.datasets import fetch_california_housing

california_housing = fetch_california_housing()

# 학습할 입력값(X : X_train, X_test 등등 변경)
california_housing_df = pd.DataFrame(  # 데이터 프레임으로 변경 후
    california_housing.data,  # 데이터 입력
    columns=california_housing.feature_names  # 컬럽값을 지정한다
)

# print(california_housing.target)

# 정답(y) - 예측해야할 주택의 가격 딕셔너리 추가
california_housing_df['MedHouseVal'] = california_housing.target
california_housing_df.head()

,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude
0,8.3252,41.0,6.984127,1.023810,322.0,2.555556,37.88,-122.23
1,8.3014,21.0,6.238137,0.971880,2401.0,2.109842,37.86,-122.22
2,7.2574,52.0,8.288136,1.073446,496.0,2.802260,37.85,-122.24
3,5.6431,52.0,5.817352,1.073059,558.0,2.547945,37.85,-122.25
4,3.8462,52.0,6.281853,1.081081,565.0,2.181467,37.85,-122.25


## 학습/평가 데이터 분리

- train_test_split: 데이터를 학습용과 평가용으로 나누어 새 데이터 성능을 확인할 준비를 함.


In [7]:
# train_test_split() : 데이터의 일부를 떄서 확인한다
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    california_housing.data,
    california_housing.target,
    test_size=0.2,
    random_state=42
)

print(X_train.shape, y_train.shape)
print(X_test.shape, y_test.shape)

(16512, 8) (16512,)
(4128, 8) (4128,)


## 회귀 평가지표
점수가 낮을수록 좋음
- MSE: 오차를 제곱해 평균낸 값으로 큰 오차에 더 민감함.
- MAE: 오차의 절댓값을 평균낸 값으로 실제 단위 해석이 쉬움.
- RMSE: MSE에 제곱근을 씌워 target과 같은 단위로 해석하는 지표임.


---

점수가 높을수록 좋음
- fit: 훈련 데이터에서 모델 또는 전처리 기준을 학습하는 메서드임.
- predict: 학습된 모델로 새 데이터의 예측값을 생성하는 메서드임.
- score: 모델의 기본 평가 점수를 계산하는 메서드임.


# 다항 feature(다항 선형) - LinearRegression 모델 평가 지표


In [15]:
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.metrics import mean_squared_error, mean_absolute_error, root_mean_squared_error
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression

# 파이프라인이란?
# 모댈생성, 학습, 변환 작업물들을 순서대로 진행하게하는 모델(메소드)
pipeline = Pipeline([
    # 1. 기존 feature를 2차 다항 feature로 확장한다.
    # 예: length, height, width가 있으면
    # length^2, length height, height^2 같은 새로운 feature가 만들어진다.
    # include_bias=False는 상수항 1을 추가하지 않겠다는 의미이다.
    ('poly', PolynomialFeatures(degree=2, include_bias=False)),

    # 2. feature들의 평균과 표준편차를 기준으로 값을 표준화한다.
    # 평균은 0, 표준편차는 1에 가깝게 맞춘다.
    # 다항 feature처럼 값의 크기가 달라질 수 있는 경우 스케일링이 중요하다.
    ('scaler', StandardScaler()),

    # 3. 변환된 feature를 사용해서 선형 회귀 모델을 학습한다.
    # fit_intercept=True는 절편을 학습하겠다는 의미이다.
    ('linear_regression', LinearRegression(fit_intercept=True)),
])

## 학습진행 -> 예측 -> 평가
# 학습 진행
# 파이프라인 전체를 학습한다.
# X_train: 입력 feature, y_train: 정답 target
# 내부적으로 다항 feature 생성 -> 스케일링 -> 선형 회귀 학습 순서로 진행된다.
pipeline.fit(X_train, y_train)

# 학습된 선형 회귀 모델만 따로 확인하고 싶을 때 꺼낸다.
model = pipeline.named_steps['linear_regression']

# 학습 데이터에 대한 예측값
# 파이프라인 내부 변환 과정을 자동으로 거친 뒤 예측한다.
y_train_pred = pipeline.predict(X_train)

# 테스트 데이터에 대한 예측값
# fit은 다시 하지 않고, 학습 때 정한 변환 기준으로 예측한다.
y_test_pred = pipeline.predict(X_test)

# 회귀 모델의 평가 결과를 표 형태로 정리한다.
ridge_eval_result = pd.DataFrame({
    # dataset: 평가 대상 데이터가 train인지 test인지 구분하는 컬럼
    'dataset': ['train', 'test'],

    # R2: 결정계수
    # 모델이 정답 y를 얼마나 잘 설명하는지 나타내는 점수
    # 1에 가까울수록 좋고, 0에 가까우면 평균으로 예측하는 것과 비슷하다.
    'R2': [
        pipeline.score(X_train, y_train),
        pipeline.score(X_test, y_test)
    ],

    # MSE: Mean Squared Error, 평균 제곱 오차
    # 실제값과 예측값의 차이를 제곱한 뒤 평균낸 값
    # 오차를 제곱하기 때문에 큰 오차에 더 민감하다.
    # 작을수록 좋다.
    'MSE': [
        mean_squared_error(y_train, y_train_pred),
        mean_squared_error(y_test, y_test_pred)
    ],

    # MAE: Mean Absolute Error, 평균 절대 오차
    # 실제값과 예측값의 차이를 절댓값으로 바꾼 뒤 평균낸 값
    # 예측값이 실제값과 평균적으로 얼마나 차이 나는지 직관적으로 볼 수 있다.
    # 작을수록 좋다.
    'MAE': [
        mean_absolute_error(y_train, y_train_pred),
        mean_absolute_error(y_test, y_test_pred)
    ],

    # RMSE: Root Mean Squared Error, 평균 제곱근 오차
    # MSE에 루트를 씌운 값
    # target과 같은 단위로 해석할 수 있다.
    # 작을수록 좋다.
    'RMSE': [
        root_mean_squared_error(y_train, y_train_pred),
        root_mean_squared_error(y_test, y_test_pred)
    ]
})

# model.coef_는 학습된 선형 회귀 모델의 회귀계수이다.
# 각 feature가 예측값에 얼마나 영향을 주는지 나타내는 가중치이다.
# 회귀계수 제곱합은 계수들이 전체적으로 얼마나 큰지 확인하는 값이다.
print('회귀계수 제곱합:', np.sum(model.coef_ ** 2))

ridge_eval_result

회귀계수 제곱합: 7170.956218225917


,dataset,R2,MSE,MAE,RMSE
0,train,0.685268,0.420727,0.460838,0.648634
1,test,0.645682,0.464302,0.467001,0.681397


### 다항 feature를 사용한 선형 회귀 모델을 활용한
### Ridge L2 규제 선형 회귀 모델 평가 지표

In [18]:
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.metrics import mean_squared_error, mean_absolute_error, root_mean_squared_error
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge

# 파이프라인이란?
# 모댈생성, 학습, 변환 작업물들을 순서대로 진행하게하는 모델(메소드)
pipeline = Pipeline([
    # 1. 기존 feature를 2차 다항 feature로 확장한다.
    # 예: length, height, width가 있으면
    # length^2, length height, height^2 같은 새로운 feature가 만들어진다.
    # include_bias=False는 상수항 1을 추가하지 않겠다는 의미이다.
    ('poly', PolynomialFeatures(degree=2, include_bias=False)),

    # 2. feature들의 평균과 표준편차를 기준으로 값을 표준화한다.
    # 평균은 0, 표준편차는 1에 가깝게 맞춘다.
    # 다항 feature처럼 값의 크기가 달라질 수 있는 경우 스케일링이 중요하다.
    ('scaler', StandardScaler()),

    # 3. 변환된 feature를 사용해서 선형 회귀 모델을 학습한다.
    # fit_intercept=True는 절편을 학습하겠다는 의미이다.

    # Ridge : 회귀계수 제곱합을 줄이는 규제
    # alpha : 구제 강도

    # 3. 변환된 feature를 사용해서 Ridge 회귀 모델을 학습한다.
    # Ridge는 선형 회귀에 L2 규제를 추가한 모델이다.
    # alpha는 규제의 강도를 조절하는 값이다.
    # alpha가 클수록 회귀계수가 작아지도록 더 강하게 제한한다.
    # alpha=100은 비교적 강한 규제를 적용하겠다는 의미이다.
    # 규제가 강하면 과대적합은 줄어들 수 있지만, 너무 크면 과소적합이 생길 수 있다.
    # 'model'은 파이프라인 안에서 이 Ridge 모델 단계를 부르는 이름이다.
    ('model', Ridge(alpha=1)),
])

## 학습진행 -> 예측 -> 평가
# 학습 진행
# 파이프라인 전체를 학습한다.
# X_train: 입력 feature, y_train: 정답 target
# 내부적으로 다항 feature 생성 -> 스케일링 -> Ridge로 변경
pipeline.fit(X_train, y_train)

# 학습된 선형 회귀 모델만 따로 확인하고 싶을 때 꺼낸다.
model = pipeline.named_steps['model']

# 학습 데이터에 대한 예측값
# 파이프라인 내부 변환 과정을 자동으로 거친 뒤 예측한다.
y_train_pred = pipeline.predict(X_train)

# 테스트 데이터에 대한 예측값
# fit은 다시 하지 않고, 학습 때 정한 변환 기준으로 예측한다.
y_test_pred = pipeline.predict(X_test)

# 회귀 모델의 평가 결과를 표 형태로 정리한다.
ridge_eval_result = pd.DataFrame({
    # dataset: 평가 대상 데이터가 train인지 test인지 구분하는 컬럼
    'dataset': ['train', 'test'],

    # R2: 결정계수
    # 모델이 정답 y를 얼마나 잘 설명하는지 나타내는 점수
    # 1에 가까울수록 좋고, 0에 가까우면 평균으로 예측하는 것과 비슷하다.
    'R2': [
        pipeline.score(X_train, y_train),
        pipeline.score(X_test, y_test)
    ],

    # MSE: Mean Squared Error, 평균 제곱 오차
    # 실제값과 예측값의 차이를 제곱한 뒤 평균낸 값
    # 오차를 제곱하기 때문에 큰 오차에 더 민감하다.
    # 작을수록 좋다.
    'MSE': [
        mean_squared_error(y_train, y_train_pred),
        mean_squared_error(y_test, y_test_pred)
    ],

    # MAE: Mean Absolute Error, 평균 절대 오차
    # 실제값과 예측값의 차이를 절댓값으로 바꾼 뒤 평균낸 값
    # 예측값이 실제값과 평균적으로 얼마나 차이 나는지 직관적으로 볼 수 있다.
    # 작을수록 좋다.
    'MAE': [
        mean_absolute_error(y_train, y_train_pred),
        mean_absolute_error(y_test, y_test_pred)
    ],

    # RMSE: Root Mean Squared Error, 평균 제곱근 오차
    # MSE에 루트를 씌운 값
    # target과 같은 단위로 해석할 수 있다.
    # 작을수록 좋다.
    'RMSE': [
        root_mean_squared_error(y_train, y_train_pred),
        root_mean_squared_error(y_test, y_test_pred)
    ]
})

# model.coef_는 학습된 선형 회귀 모델의 회귀계수이다.
# 각 feature가 예측값에 얼마나 영향을 주는지 나타내는 가중치이다.
# 회귀계수 제곱합은 계수들이 전체적으로 얼마나 큰지 확인하는 값이다.
print('회귀계수 제곱합:', np.sum(model.coef_ ** 2))

ridge_eval_result

# L2(Ridge) 규제는 다항 feature의 표현력을 높이면서 alpha로 회귀계수를 제한
# alpha값에 따라 R2, MSE 값들이 달라진다


회귀계수 제곱합: 119.22218689328248


,dataset,R2,MSE,MAE,RMSE
0,train,0.669386,0.441958,0.482729,0.664799
1,test,0.639133,0.472883,0.487971,0.687665


### 최적의 alpha값 찾기


## 교차검증

- cross_val_score: 여러 fold의 검증 점수를 계산해 평균 성능을 더 안정적으로 확인함.
- cross_val_score() : 학습데이터를 여러 fold(데이터들을 접어서 구간을 나눔)로 나눠 검증 점수를 계산



In [17]:
from sklearn.model_selection import cross_val_score

# cross_val_score() : 학습데이터를 여러 fold(데이터들을 접어서 구간을 나눔)로 나눠 검증 점수를 계산

# 테스트해볼 alpha값 준비
alphas = [0, 0.1, 1, 10, 200, 300]
# 검증좀수를 모아둘 리스트
# 각 alpha 값마다 교차 검증으로 나온 평균 MSE를 저장할 리스트이다.
cv_results = []

# alpha 값을 하나씩 바꿔가며 반복한다.
for alpha in alphas:
    # 현재 alpha 값을 사용하는 파이프라인을 새로 만든다.
    # alpha마다 Ridge 모델이 달라지므로 반복문 안에서 pipeline을 다시 생성한다.
    pipeline = Pipeline([
        # 1. 기존 feature를 2차 다항 feature로 확장한다.
        ('poly', PolynomialFeatures(degree=2, include_bias=False)),

        # 2. 확장된 feature들의 단위를 평균 0, 표준편차 1 기준으로 맞춘다.
        ('scaler', StandardScaler()),

        # 3. 현재 alpha 값을 사용하는 Ridge 회귀 모델을 만든다.
        ('model', Ridge(alpha=alpha))
    ])

    # cross_val_score는 cv=5이므로 학습 데이터를 5등분한다.
    # 그중 4개 조각으로 학습하고, 나머지 1개 조각으로 검증한다.
    # 이 과정을 검증 조각을 바꿔가며 총 5번 반복한다.

    # scoring='neg_mean_squared_error'
    # sklearn은 점수가 클수록 좋은 방향으로 통일하기 위해
    # MSE에 음수(-)를 붙인 값을 반환한다.
    # 하지만 우리가 해석할 때는 양수 MSE가 편하므로 -1을 곱해서 다시 양수로 바꾼다.
    scores = -1 * cross_val_score(
        pipeline,  # 평가할 모델 파이프라인
        X_train,  # 교차 검증에 사용할 입력 feature
        y_train,  # 교차 검증에 사용할 정답 target
        cv=5,  # 데이터를 5개 fold로 나누어 검증(5개의 데이터 조각)
        scoring='neg_mean_squared_error'  # 데이터를 나누어 음수로 표시한다 # 평가 기준: MSE
    )

    # 현재 alpha에서 나온 5번의 MSE 평균을 저장한다.
    # 평균 MSE가 작을수록 더 좋은 alpha라고 볼 수 있다.
    cv_results.append({
        'alpha': alpha,  # 알파값은 얼마고
        'mean_MSE': scores.mean(),  # 평균은 얼마고
        'std_MSE': scores.std()  # 표준편차는 얼마인가
    })

ridge_cv_results = pd.DataFrame(cv_results)
ridge_cv_results



,alpha,mean_MSE,std_MSE
0,0.0,10.448255,18.601131
1,0.1,1.812390,2.335427
2,1.0,1.005593,0.782599
3,10.0,4.019982,6.748552
4,200.0,1.658760,2.318480
5,300.0,1.249920,1.498657


## L1
- L1규제방식을 구현한 Lasso클래스를 사용할 수 있다.
- alpha값을 통해 특정 회귀계수를 0까지 제한, 특정속성을 회귀계산에서 배제하는 것도 가능.


In [30]:
# alpha값이 커질수록 강햔 규제가 적용되어 feature가 많이 사라짐
from sklearn.linear_model import Lasso

# cross_val_score() : 학습데이터를 여러 fold(데이터들을 접어서 구간을 나눔)로 나눠 검증 점수를 계산


# Lasso는 선형 회귀에 L1 규제를 추가한 모델이다.
# L1 규제는 중요하지 않은 feature의 회귀계수를 0으로 만들 수 있다.
# 그래서 Lasso는 feature 선택 효과가 있다.

# alpha값이 커질수록 강한 규제가 적용된다.
# 규제가 강해지면 더 많은 feature의 계수가 0이 될 수 있다.
alphas = [0.003, 0.07, 0.1, 0.5, 1]

# alpha별 feature 계수를 저장할 표이다.
# 각 열에는 특정 alpha에서 학습된 회귀계수가 들어간다.
coef_df = pd.DataFrame()

# alpha별 평가 결과를 저장할 리스트이다.
lass_results = []

for alpha in alphas:
    # alpha 값을 하나씩 바꿔가며 Lasso 모델을 만든다.
    pipeline = Pipeline([
        # 기존 feature를 2차 다항 feature로 확장한다.
        ('poly', PolynomialFeatures(degree=2, include_bias=False)),

        # feature들의 단위를 평균 0, 표준편차 1 기준으로 맞춘다.
        ('scaler', StandardScaler()),

        # Lasso 회귀 모델을 만든다.
        # alpha는 규제 강도이다.
        # max_iter는 반복 학습 횟수를 늘려 수렴 경고를 줄이기 위해 사용한다.
        # ('lasso', Lasso(alpha=alpha, max_iter=10000)),
        ('model', Lasso(alpha=alpha, max_iter=20000)),
    ])

    # 학습 데이터로 파이프라인 전체를 학습한다.
    # 실행 순서: 다항 feature 생성 -> 스케일링 -> Lasso 학습
    pipeline.fit(X_train, y_train)

    # 파이프라인 안에서 학습된 Lasso 모델을 꺼낸다.
    # 위에서 단계 이름을 'lasso'로 만들었기 때문에 같은 이름으로 접근한다.
    lasso_model = pipeline.named_steps['model']

    # 다항 변환 후 만들어진 feature 이름을 가져온다.
    # X_train이 DataFrame이면 컬럼명을 사용하고,
    # numpy array이면 x0, x1, x2 같은 기본 이름을 사용한다.
    if hasattr(X_train, "columns"):
        feature = pipeline.named_steps['poly'].get_feature_names_out(X_train.columns)
    else:
        feature = pipeline.named_steps['poly'].get_feature_names_out()

    # 각 alpha에서 학습된 feature별 회귀계수를 저장한다.
    # 계수가 0이면 해당 feature는 Lasso에 의해 거의 사용되지 않는다고 볼 수 있다.
    coef_df[f'alpha_{alpha}'] = pd.Series(
        lasso_model.coef_,
        index=feature
    )

    # 테스트 데이터에 대한 예측값을 만든다.
    y_test_pred = pipeline.predict(X_test)

    # 현재 alpha의 평가 결과를 저장한다.
    lass_results.append({
        # 현재 사용한 alpha 값
        'alpha': alpha,

        # MSE: 실제값과 예측값의 차이를 제곱해서 평균낸 값
        # 작을수록 좋다.
        'MSE': mean_squared_error(y_test, y_test_pred),

        # R2: 모델이 정답을 얼마나 잘 설명하는지 나타내는 점수
        # 1에 가까울수록 좋다.
        'R2': pipeline.score(X_test, y_test),

        # 회귀계수가 0에 가까운 feature 개수
        # 이 값이 클수록 Lasso가 많은 feature를 제거한 것이다.
        'zero_coef_count': np.sum(np.isclose(lasso_model.coef_, 0)),
    })

# 리스트로 모은 평가 결과를 표로 변환한다.
lass_result_df = pd.DataFrame(lass_results)
lass_result_df

# Lasso는 alpha가 커질수록 규제가 강해져 0이되는 계수의 수가 증가한다
# -> feature가 많이 제거되어 예측 성능이 하락할수있따

,alpha,MSE,R2,zero_coef_count
0,0.003,0.505882,0.613951,20
1,0.070,0.659940,0.496386,41
2,0.100,0.669386,0.489178,41
3,0.500,0.937914,0.284259,43
4,1.000,1.310696,-0.000219,44


## 회귀 평가지표

- MSE: 오차를 제곱해 평균낸 값으로 큰 오차에 더 민감함.
- fit: 훈련 데이터에서 모델 또는 전처리 기준을 학습하는 메서드임.
- predict: 학습된 모델로 새 데이터의 예측값을 생성하는 메서드임.


## L1 + L2
L1규제, L2규제를 적절한 비율로 모두 적용하는 ElasticNet 선형회귀모델을 사용할 수 있다.
ElasticNet은 **회귀 분석** 기법 중 하나로, **Lasso**와 **Ridge**의 규제를 결합한 모델이다.
Lasso는 특성 선택에 효과적이고, Ridge는 모든 특성을 다루면서 모델을 규제한다.
ElasticNet은 이 두 가지 규제(L1과 L2)를 적절히 혼합하여 사용하는 방법이다.
**alpha 파라미터**
alpha는 a + b를 의미한다.
- a는 L1규제용 alpha값이다.
- b는 L2규제용 alpha값이다.
**l1_ratio 파라미터**
L1규제용 alpha값의 비율이다. $\frac{a}{a + b}$
- alpha가 10이고, l1_ratio가 0.7이면 a = 7, b = 3이다.
- alpha가 10이고, l1_ratio가 1이면 a = 10, b = 0이다. 즉, L1규제만 사용한다.
- alpha가 10이고, l1_ratio가 0이면 a = 0, b = 10이다. 즉, L2규제만 사용한다.
**수식:**
$$J(β) = RSS + α [ λ * ||β||₁ + (1 - λ) * ||β||₂² ]$$
- **RSS**: Residual Sum of Squares (예측 오차)
- **α**: 전체 규제 강도 (크면 규제가 강해짐)
- **λ**: L1과 L2 규제의 비율 조절 (0 ≤ λ ≤ 1)
  - λ = 1 → Lasso만 적용
  - λ = 0 → Ridge만 적용
- **||β||₁**: L1 노름 (∑|βᵢ|), 특성 선택
- **||β||₂²**: L2 노름 제곱 (∑βᵢ²), 계수 축소
**특징:**
- **Lasso와 Ridge의 장점을 결합**: ElasticNet은 Lasso의 **특성 선택** 능력과 Ridge의 **강한 규제** 특성을 모두 반영한다.
- **고차원 데이터에 적합**: 상관관계가 높은 특성이 많은 데이터나 차원이 높은 데이터에 적합하다.
- **Overfitting 방지**: 두 가지 규제를 혼합하여 과적합을 효과적으로 방지할 수 있다.


## 모델 학습

- fit: 훈련 데이터에서 모델 또는 전처리 기준을 학습하는 메서드임.
- score: 모델의 기본 평가 점수를 계산하는 메서드임.


## 결과 확인

- 실행 결과: 앞에서 만든 객체와 실행 결과를 확인하며 다음 단계로 연결함.


## 다중공선성 MultiCollinearity
특성간의 상관관계가 너무 높은 경우를 가리킨다.
주택데이터에서 면적특성과 방의크기특성은 높은 상관관계(상관계수 0.8이상)를 가질수 있다.
다중공선성특성에 대한 회귀계수가 크게 학습이 되고, 이는 특정데이터에 민감한 과대적합을 유발한다.
**해결책**
- 다중공선성 특성 제거
- 규제모델을 사용한 회귀계수 억제


## 결과 확인

- 실행 결과: 앞에서 만든 객체와 실행 결과를 확인하며 다음 단계로 연결함.


## 학습/평가 데이터 분리

- train_test_split: 데이터를 학습용과 평가용으로 나누어 새 데이터 성능을 확인할 준비를 함.
- MSE: 오차를 제곱해 평균낸 값으로 큰 오차에 더 민감함.
- fit: 훈련 데이터에서 모델 또는 전처리 기준을 학습하는 메서드임.
- predict: 학습된 모델로 새 데이터의 예측값을 생성하는 메서드임.
